In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import sys; sys.path.insert(0, '..')
from utils import Conv, C3k2, SPPF, Bottleneck

## <strong>C2PSA Block</strong>

<h2><strong>Role of the C2PSA Block (Convolutional and Attention-Based Block)</strong></h2>

The C2PSA block enhances feature extraction and processing capabilities through:

- **<u>Dual-Path Processing:</u>** Splits input features into two paths—one for direct convolutional processing and the other for attention-based transformations via PSABlock modules. This design captures both local and global feature dependencies.

- **<u>Attention Mechanisms:</u>**
Utilizes multi-head self-attention (via the PSABlock) to focus on the most relevant spatial features, improving the network’s ability to model long-range dependencies and subtle feature interactions.

- **<u>Feature Fusion:</u>**
Combines the outputs of convolutional and attention pathways to produce a richer, more expressive feature map, enhancing the network’s capability to detect complex patterns.

- **<u>Feed-Forward Enhancements:</u>**
Integrates lightweight feed-forward layers within the PSABlock to refine features, ensuring efficient and precise information propagation.


By leveraging convolutional and attention mechanisms synergistically, the C2PSA block improves feature representation, enabling better detection accuracy and robustness.

In [ ]:
class Attention(nn.Module):
    '''
    Attention module that performs self-attention on the input tensor.
    
    Args:
        dim (int): The input tensor dimension.
        num_heads (int): The number of attention heads.
        attn_ratio (float): The ratio of the attention key dimension to the head dimension.

    Attributes:
        num_heads (int): The number of attention heads.
        head_dim (int): The dimension of each attention head.
        key_dim (int): The dimension of the attention key.
        scale (float): The scaling factor for the attention scores.
        qkv (Conv): Convolutional layer for computing the query, key, and value.
        proj (Conv): Convolutional layer for projecting the attended values.
        pe (Conv): Convolutional layer for positional encoding.
    
    '''
    def __init__(self, dim, num_heads = 8, attn_ratio = 0.5):
        '''
        Initializes the multi-head attention module with query, key, value projections and positional encoding.
        
        '''
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.key_dim = int(self.head_dim * attn_ratio)
        self.scale = self.key_dim ** -0.5
        nh_kd = self.key_dim * num_heads
        h = dim + nh_kd * 2
        self.qkv = Conv(dim, h, 1, act= False)
        self.proj = Conv(dim, dim, 1, act = False)
        self.pe = Conv(dim, dim, 3, 1, g = dim, act = False)
        
    def forward(self, x):
        '''
        Computes the self-attention on the input tensor and returns the attended output.
        
        Args:
            x (torch.Tensor): The input tensor of shape (B, C, H, W).
        Returns:
            torch.Tensor: The output tensor after applying self-attention, of shape (B, C, H, W).
        '''
        B, C, H, W = x.shape
        N = H * W
        qkv = self.qkv(x)
        q, k, v = qkv.view(B, self.num_heads, self.key_dim*2 + self.head_dim, N).split(
            [self.key_dim, self.key_dim, self.head_dim], dim = 2
        )
        
        attn = (q.transpose(-2, -1) @ k) * self.scale
        attn = attn.softmax(dim = -1)
        x = (v @ attn.transpose(-2, -1)).view(B, C, H, W) + self.pe(v.reshape(B, C, H, W))
        x = self.proj(x)
        return x

In [13]:
class PSABlock(nn.Module):
    '''This class encapsulates the functionality for applying multi-head attention and feed-forward neural network with optional shortcut connectoin'''
    
    def __init__(self, c, attn_ratio = 0.5, num_heads = 4, shortcut = True) -> None:
        super().__init__()
        
        self.attn = Attention(c, attn_ratio = attn_ratio, num_heads = num_heads) # initialize the attention mechanism with specified parameters
        self.ffn = nn.Sequential(
            Conv(c, c*2, 1), Conv(c*2, c, 1, act = False)
        )
        self.add = shortcut
        
    def forward(self, x):
        """Executes a forward pass through the PSABlock, applying attention and feed-forward operations, and optionally adding a shortcut connection."""
        
        x = x + self.attn(x) if self.add else self.attn(x) # apply attention and add input to output if shortcut is enabled
        x = x + self.ffn(x) if self.add else self.ffn(x)

        return x
    
class C2PSA(nn.Module):
    '''
    This module implements a convolutional block with attention mechanism to enhance
    feature representation. It consists of a convolutional layer followed by a channel-wise
    '''
    
    def __init__(self, c1, c2, n = 1, e = 0.5):
        # initialize the C2SPA module with input channels c1, output channels c2, number of convolutional layers n, and expansion ratio e
        super().__init__()
        assert c1 == c2, "Input and output channels must be the same for C2PSA"
        self.c = int(c1 * e) # calculate the intermediate channel size based on the expansion ratio
        self.cv1 = Conv(c1, 2*self.c, 1)
        self.cv2 = Conv(2*self.c, c1, 1)
        
        self.m = nn.Sequential(
            *(PSABlock(self.c, attn_ratio = 0.5, num_heads = self.c // 64) for _ in range(n))
        )
    
    def forward(self, x):
        """Executes a forward pass through the C2PSA module, applying convolutional transformations and attention mechanisms to enhance feature representation."""
        
        a, b = self.cv1(x).split((self.c, self.c), dim = 1) # split the output of the first convolutional layer into two parts
        b = self.m(b)  # apply the attention mechanism to one part of the split output
        return self.cv2(torch.cat((a,b), dim = 1)) # concatenate the two parts and apply the second convolutional layer to produce the final output

<h1><strong>Upsample Layer</strong></h1>

<h2><strong>Role of the Upsample Layer</strong></h2>

This module enhances feature map resolution through:

- **<u>Scale Factor:</u>** Increases the spatial dimensions of the input feature map by a factor of 2.0, enabling finer spatial detail recovery.

- **<u>Interpolation Mode:</u>**
Utilizes the 'nearest' interpolation method to replicate pixel values, ensuring computational efficiency and simplicity.

This layer is essential for tasks like object detection, where higher resolution is required to improve localization accuracy.

<h1><strong> Concat Layer </strong></h1>

<h2><strong>Role of the Concat Layer</strong></h2>

This module combines feature maps from multiple layers through:

- **<u>Tensor Concatenation:</u>**
Merges a list of tensors along a specified dimension (default: dimension 1, corresponding to the channel dimension), enabling the integration of features from different layers.

- **<u>Flexible Input:</u>**
Supports concatenation of tensors from the current and earlier layers, providing adaptability in network design and feature flow.

This layer plays an important role in deep learning models by allowing the fusion of diverse feature representations, enhancing the network's capacity to capture complex patterns.

In [16]:
class Concat(nn.Module):
    '''
    Concatenate a list of tensors along dimension
    '''
    def __init__(self, dimension = 1):
        super().__init__()
        self.d = dimension
        
    def forward(self, x):
        return torch.cat(x, dim = self.d)